# Step 1 & 2 — Data Acquisition, Loading & Cleaning
Load the master dataset, inspect quality, standardise units, and clean.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/ai_datacenter_master.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Shape: (50, 15)

Columns: ['company', 'year', 'electricity_gwh', 'it_load_gwh', 'water_litres', 'water_consumed_litres', 'pue', 'wue', 'cue', 'co2_tons', 'renewable_pledged_pct', 'renewable_actual_pct', 'queries_million', 'region', 'facility_count']


,company,year,electricity_gwh,it_load_gwh,water_litres,water_consumed_litres,pue,wue,cue,co2_tons,renewable_pledged_pct,renewable_actual_pct,queries_million,region,facility_count
0,Google,2015,7.81,6.99,1858291968,1080046220,1.119,266.000,393.7010,3076671,37,34,811.2,Europe,15
1,Google,2016,9.04,8.13,2070690052,1214180159,1.113,254.845,294.5228,2663697,44,40,900.4,Europe,17
2,Google,2017,9.49,8.55,2143575418,1304213297,1.110,250.733,273.4351,2594650,56,50,1074.7,Europe,19
3,Google,2018,10.68,9.58,2348720224,1570079339,1.114,245.090,226.7284,2420469,67,58,1280.6,North America,21
4,Google,2019,12.77,11.52,2747123118,1577968526,1.108,238.386,190.4309,2431927,73,61,1509.6,Asia-Pacific,23


In [2]:
print("=== Data Types ===")
print(df.dtypes)
print(f"\n=== Null Counts ===")
print(df.isnull().sum())
print(f"\n=== Descriptive Stats ===")
df.describe()

=== Data Types ===
company                   object
year                       int64
electricity_gwh          float64
it_load_gwh              float64
water_litres               int64
water_consumed_litres      int64
pue                      float64
wue                      float64
cue                      float64
co2_tons                   int64
renewable_pledged_pct      int64
renewable_actual_pct       int64
queries_million          float64
region                    object
facility_count             int64
dtype: object

=== Null Counts ===
company                  0
year                     0
electricity_gwh          0
it_load_gwh              0
water_litres             0
water_consumed_litres    0
pue                      0
wue                      0
cue                      0
co2_tons                 0
renewable_pledged_pct    0
renewable_actual_pct     0
queries_million          0
region                   0
facility_count           0
dtype: int64

=== Descriptive Stats ===


,year,electricity_gwh,it_load_gwh,water_litres,water_consumed_litres,pue,wue,cue,co2_tons,renewable_pledged_pct,renewable_actual_pct,queries_million,facility_count
count,50.000000,50.000000,50.000000,5.000000e+01,5.000000e+01,50.00000,50.000000,50.000000,5.000000e+01,50.000000,50.000000,50.00000,50.000000
mean,2019.500000,107210.471200,67857.169600,4.536628e+11,3.023380e+11,1.24234,334.364580,216.334648,2.232659e+06,66.440000,51.980000,5116.59800,2079.660000
std,2.901442,225182.273223,142519.240493,9.441266e+11,6.294679e+11,0.15604,199.226583,192.347380,1.553629e+06,27.883987,22.896832,10708.01188,4177.391827
min,2015.000000,2.380000,1.910000,1.136262e+09,7.819612e+08,1.09200,1.638000,0.379500,1.680000e+05,20.000000,15.000000,51.60000,6.000000
25%,2017.000000,8.882500,7.362500,2.840869e+09,1.798488e+09,1.11950,228.113000,83.680575,8.822128e+05,40.500000,32.500000,449.22500,21.250000
50%,2019.500000,13.355000,11.900000,4.495392e+09,2.959186e+09,1.18000,413.287000,175.470250,2.392658e+06,66.000000,50.000000,1134.40000,36.000000
75%,2022.000000,26.245000,22.605000,8.459675e+09,5.298209e+09,1.28325,498.473250,297.807575,3.348954e+06,100.000000,71.000000,2764.70000,83.000000
max,2024.000000,850000.000000,537974.680000,3.570000e+12,2.380000e+12,1.57800,596.177000,726.276000,5.231975e+06,100.000000,94.000000,53022.50000,12500.000000


In [3]:
print("Companies:", df['company'].unique())
print("Years:", sorted(df['year'].unique()))
print("Regions:", df['region'].unique())
print(f"\nRows per company:")
print(df['company'].value_counts())

Companies: ['Google' 'Microsoft' 'Meta' 'AWS' 'All']
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Regions: ['Europe' 'North America' 'Asia-Pacific' 'Global']

Rows per company:
company
Google       10
Microsoft    10
Meta         10
AWS          10
All          10
Name: count, dtype: int64


## Data Cleaning

In [4]:
# Validate PUE range (1.0 to 2.5 is physically realistic)
before = len(df)
df = df[df['pue'].between(1.0, 2.5)]
print(f"PUE filter: {before} -> {len(df)} rows ({before - len(df)} removed)")

# Drop duplicates
before = len(df)
df = df.drop_duplicates(subset=['company', 'year'])
print(f"Dedup: {before} -> {len(df)} rows ({before - len(df)} removed)")

# Ensure no negative values in resource columns
resource_cols = ['electricity_gwh', 'it_load_gwh', 'water_litres', 'water_consumed_litres', 'co2_tons']
for col in resource_cols:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f"Fixing {neg} negative values in {col}")
        df[col] = df[col].abs()

print(f"\nCleaned dataset: {df.shape[0]} rows")

PUE filter: 50 -> 50 rows (0 removed)
Dedup: 50 -> 50 rows (0 removed)

Cleaned dataset: 50 rows


In [5]:
# Save cleaned dataset
df.to_csv('../data/ai_datacenter_master_clean.csv', index=False)
print("Saved: data/ai_datacenter_master_clean.csv")
df.head()

Saved: data/ai_datacenter_master_clean.csv


,company,year,electricity_gwh,it_load_gwh,water_litres,water_consumed_litres,pue,wue,cue,co2_tons,renewable_pledged_pct,renewable_actual_pct,queries_million,region,facility_count
0,Google,2015,7.81,6.99,1858291968,1080046220,1.119,266.000,393.7010,3076671,37,34,811.2,Europe,15
1,Google,2016,9.04,8.13,2070690052,1214180159,1.113,254.845,294.5228,2663697,44,40,900.4,Europe,17
2,Google,2017,9.49,8.55,2143575418,1304213297,1.110,250.733,273.4351,2594650,56,50,1074.7,Europe,19
3,Google,2018,10.68,9.58,2348720224,1570079339,1.114,245.090,226.7284,2420469,67,58,1280.6,North America,21
4,Google,2019,12.77,11.52,2747123118,1577968526,1.108,238.386,190.4309,2431927,73,61,1509.6,Asia-Pacific,23
